In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

In [12]:
# load users data
user_df = pd.read_csv(
    RAW_DATA_DIR / "users.csv"
)

print(user_df.head())

   code company             name  gender  age
0     0    4You        Roy Braun    male   21
1     1    4You   Joseph Holsten    male   37
2     2    4You    Wilma Mcinnis  female   48
3     3    4You     Paula Daniel  female   23
4     4    4You  Patricia Carson  female   44


In [14]:
# dataset info
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   code     1340 non-null   int64 
 1   company  1340 non-null   object
 2   name     1340 non-null   object
 3   gender   1340 non-null   object
 4   age      1340 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 52.5+ KB


In [15]:
user_df.describe(include="all")

,code,company,name,gender,age
count,1340.000000,1340,1340,1340,1340.000000
unique,NaN,5,1338,3,NaN
top,NaN,4You,Charlotte Johnson,male,NaN
freq,NaN,453,2,452,NaN
mean,669.500000,NaN,NaN,NaN,42.742537
std,386.968991,NaN,NaN,NaN,12.869779
min,0.000000,NaN,NaN,NaN,21.000000
25%,334.750000,NaN,NaN,NaN,32.000000
50%,669.500000,NaN,NaN,NaN,42.000000
75%,1004.250000,NaN,NaN,NaN,54.000000


In [16]:
print("Shape :", user_df.shape)

Shape : (1340, 5)


In [17]:
user_df.columns

Index(['code', 'company', 'name', 'gender', 'age'], dtype='object')

In [18]:
# missing values
user_df.isnull().sum()

code       0
company    0
name       0
gender     0
age        0
dtype: int64

In [19]:
(user_df.isnull().sum()/len(user_df))*100

code       0.0
company    0.0
name       0.0
gender     0.0
age        0.0
dtype: float64

In [21]:
# duplicate records
print(
    "Duplicate Records :",
    user_df.duplicated().sum()
)

Duplicate Records : 0


In [16]:
# check shape of the dataset
print(user_ml.shape)

(1340, 3)


In [22]:
# prepare features and target variable
user_df["first_name"] = (
    user_df["name"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
)

X = user_df[
    ["first_name", "company", "age"]
]

y = user_df["gender"]

X.head()

,first_name,company,age
0,Roy,4You,21
1,Joseph,4You,37
2,Wilma,4You,48
3,Paula,4You,23
4,Patricia,4You,44


In [23]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (1072, 3)
Testing : (268, 3)


In [24]:
# built the model pipeline
preprocessor = ColumnTransformer(
    transformers=[

        (
            "name",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(2, 4)
            ),
            "first_name"
        ),

        (
            "company",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            ["company"]
        ),

        (
            "age",
            StandardScaler(),
            ["age"]
        )
    ]
)


gender_model = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [25]:
# train
gender_model.fit(
    X_train,
    y_train
)

print("Gender model trained successfully.")

Gender model trained successfully.


In [26]:
# evaluate the model
y_pred = gender_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy: 0.5634328358208955

Classification Report:
              precision    recall  f1-score   support

      female       0.64      0.62      0.63        90
        male       0.61      0.76      0.67        90
        none       0.39      0.31      0.34        88

    accuracy                           0.56       268
   macro avg       0.55      0.56      0.55       268
weighted avg       0.55      0.56      0.55       268



In [27]:
model_path = MODELS_DIR / "gender_classifier.pkl"

joblib.dump(
    gender_model,
    model_path
)

print(
    f"Model saved successfully at: {model_path}"
)

Model saved successfully at: d:\DL Project\New folder\Travel_Analytics_MLOps\models\gender_classifier.pkl


In [ ]:
test_data = pd.DataFrame([
    {
        "first_name": "Patricia",
        "company": "4You",
        "age": 44
    }
])

gender_model.predict(test_data)

array(['female'], dtype=object)